# Call the HighRev model API

L06-E5 and L06-E6. Start your FastAPI service using the lesson instructions, then run these cells. This notebook is only a client: it does not import PyCaret, load a model or start a server. Keep the API terminal open. You need `requests` in the notebook environment.

A successful request proves the HTTP interface works. It does not prove future-revenue prediction validity or production hosting readiness.


In [ ]:
import json
import requests

base_url = "http://127.0.0.1:8016"
response = requests.get(base_url + "/health", timeout=10)
response.raise_for_status()
first_model_uri = response.json()["model_uri"]
print(response.json())


## Send prepared features

No target label or customer ID belongs in the feature payload. `null` (Python `None`) means a missing value; omitting a required key is a schema error. The saved pipeline handles preprocessing.


In [ ]:
payload = {
    "rows": [{
        "pages_visited": 8.0,
        "age_first_order": 35.0,
        "gender": "F",
        "campaign": True,
        "ip_address_country": "Poland",
        "browser": "Chrome",
        "os": "Windows",
    }]
}
response = requests.post(base_url + "/predict", json=payload, timeout=30)
response.raise_for_status()
prediction = response.json()
assert len(prediction["probabilities"]) == len(payload["rows"])
assert all(0 <= value <= 1 for value in prediction["probabilities"])
print(json.dumps(prediction, indent=2))


## Check an intentional failure

Explain the status and error location. Why should the target be rejected rather than silently ignored?


In [ ]:
invalid = {"rows": [{**payload["rows"][0], "high_revenue": True}]}
response = requests.post(base_url + "/predict", json=invalid, timeout=10)
assert response.status_code == 422
print(response.status_code)
print(json.dumps(response.json(), indent=2))


## After rollback and server restart

Pause here. Follow L06-E6: restore a previously approved version, stop the API, set its immutable model URI and restart. Then rerun a request below. If there was only one approved version, explain why an actual version change cannot be demonstrated rather than inventing approval. No target labels or final-test scores are used in this client.


In [ ]:
response = requests.post(base_url + "/predict", json=payload, timeout=30)
response.raise_for_status()
restored = response.json()
print("Initially served:", first_model_uri)
print("Now served:", restored["model_uri"])
print("Probabilities:", restored["probabilities"])
# Write down whether the served URI matches the rollback decision.


Stop the server with Ctrl+C in its terminal when finished. Keep the request and response with your observations. Continue to the one-time final audit only after freezing the model choice.
